# Evaluation

1. Retrieval evaluation (Recall@k)
2. Embedding visualization (t‑SNE)
3. Baseline (Qwen) evaluation & visualization
4. Inference helper (embedding & retrieval)

In [1]:
import sys, os
if "src" not in sys.path:
    sys.path.append(os.path.abspath("src"))

import json
import numpy as np

from src.graph_builder import SPARQLGraphBuilder
from src.semantic_dataset import SemanticSPARQLDataset
from src.inference import SPARQLEmbedder


/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/ganesh-jupyterhub/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SEMANTIC_PATH = "dataset/semantic_triplets_dbpedia.json"

with open(SEMANTIC_PATH, "r", encoding="utf-8") as f:
    raw_sem = json.load(f)

with open("dataset/splits/semantic_triplet_splits_dbpedia.json", "r", encoding="utf-8") as f:
    split_info = json.load(f)

test_idx = split_info["test_idx"]
print(f"Loaded {len(raw_sem)} semantic entries | Test size: {len(test_idx)}")


Loaded 3858 semantic entries | Test size: 387


In [3]:
anchors = []
positives_per_item = []

for idx in test_idx:
    item = raw_sem[idx]
    anchors.append(item["anchor_query"])
    positives_per_item.append({p["query"] for p in item["positive_queries"]})

print(f"Anchors: {len(anchors)} | Positive sets: {len(positives_per_item)}")


Anchors: 387 | Positive sets: 387


In [4]:
candidate_queries = set()

for idx in test_idx:
    item = raw_sem[idx]
    candidate_queries.add(item["anchor_query"])
    for p in item["positive_queries"]:
        candidate_queries.add(p["query"])
    for n in item["negative_queries"]:
        candidate_queries.add(n["query"])

candidate_queries = list(candidate_queries)
print(f"Candidate pool size: {len(candidate_queries)}")


Candidate pool size: 7003


In [5]:
from sentence_transformers import SentenceTransformer

baseline = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")
print("Loaded Qwen/Qwen3-Embedding-0.6B baseline.")


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 355.80it/s, Materializing param=norm.weight]                              


Loaded Qwen/Qwen3-Embedding-0.6B baseline.


In [6]:
BATCH = 32
cand_embeddings = []
for i in range(0, len(candidate_queries), BATCH):
    batch_qs = candidate_queries[i : i + BATCH]
    emb = baseline.encode(batch_qs, convert_to_numpy=True, normalize_embeddings=True)
    cand_embeddings.append(emb)

cand_embeddings = np.vstack(cand_embeddings)
print("Encoded candidate pool (baseline).")


Encoded candidate pool (baseline).


### Baseline (Qwen3-0.6B) Evaluation

In [7]:
def prf_at_k(topk_queries, pos_set, k):
    """Compute precision@k, recall@k, f1@k for one query."""
    if not pos_set:
        return 0.0, 0.0, 0.0

    hits = len(topk_queries & pos_set)
    precision = hits / k
    recall = hits / len(pos_set)

    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = 2 * precision * recall / (precision + recall)

    return precision, recall, f1

def eval_baseline_at_k(k):
    p_sum, r_sum, f1_sum = 0.0, 0.0, 0.0
    total = 0

    for anchor_q, pos_set in zip(anchors, positives_per_item):
        if not anchor_q or not pos_set:
            continue

        a_emb = baseline.encode([anchor_q], convert_to_numpy=True, normalize_embeddings=True)[0]
        sims = cand_embeddings @ a_emb
        topk_idx = np.argsort(-sims)[:k]
        topk_queries = {candidate_queries[i] for i in topk_idx}

        p, r, f1 = prf_at_k(topk_queries, pos_set, k)
        p_sum += p
        r_sum += r
        f1_sum += f1
        total += 1

    if total == 0:
        return 0.0, 0.0, 0.0

    return p_sum/total, r_sum/total, f1_sum/total


for k in [3, 5, 10, 20]:
    p, r, f1 = eval_baseline_at_k(k)
    print(f"Baseline @ {k:>2} | P: {p:.4f} | R: {r:.4f} | F1: {f1:.4f}")


Baseline @  3 | P: 0.1197 | R: 0.1595 | F1: 0.1047
Baseline @  5 | P: 0.1070 | R: 0.1841 | F1: 0.1008
Baseline @ 10 | P: 0.0798 | R: 0.2221 | F1: 0.0879
Baseline @ 20 | P: 0.0633 | R: 0.2723 | F1: 0.0816


### SPARQLEncoder Evaluation

In [8]:
ckpt_path = "checkpoints/final_model_semantic-20.pt"
embedder = SPARQLEmbedder(ckpt_path)
print(f"Loaded semantic model from {ckpt_path}")

BATCH = 32
cand_embs = []
for i in range(0, len(candidate_queries), BATCH):
    batch_qs = candidate_queries[i : i + BATCH]
    emb = embedder.encode(batch_qs)  # numpy (b, d)
    cand_embs.append(emb)

cand_embs = np.vstack(cand_embs)
cand_embs = cand_embs / np.linalg.norm(cand_embs, axis=1, keepdims=True)
print("Encoded candidate pool (your model).")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 378.66it/s, Materializing param=pooler.dense.weight]                               


Loading model from checkpoints/final_model_semantic-20.pt...
Loaded semantic model from checkpoints/final_model_semantic-20.pt


/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/ganesh-jupyterhub/lib/python3.10/site-packages/torch_geometric/data/data.py:187: UserWarning: Unable to accurately infer 'num_nodes' from the attribute set '{'edge_attr_mask', 'edge_index', 'x_mask', 'x_ids', 'edge_attr_ids'}'. Please explicitly set 'num_nodes' as an attribute of 'data' to suppress this warning
  return sum([v.num_nodes for v in self.node_stores])
/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/ganesh-jupyterhub/lib/python3.10/site-packages/torch_geometric/data/collate.py:142: UserWarning: Unable to accurately infer 'num_nodes' from the attribute set '{'edge_attr_mask', 'edge_index', 'x_mask', 'x_ids', 'edge_attr_ids'}'. Please explicitly set 'num_nodes' as an attribute of 'data' to suppress this warning
  repeats = [store.num_nodes or 0 for store in stores]


Encoded candidate pool (your model).


In [9]:
def eval_model_at_k(k):
    p_sum, r_sum, f1_sum = 0.0, 0.0, 0.0
    total = 0

    for anchor_q, pos_set in zip(anchors, positives_per_item):
        if not anchor_q or not pos_set:
            continue

        a_emb = embedder.encode([anchor_q])[0]
        a_emb = a_emb / np.linalg.norm(a_emb)

        sims = cand_embs @ a_emb
        topk_idx = np.argsort(-sims)[:k]
        topk_queries = {candidate_queries[i] for i in topk_idx}

        p, r, f1 = prf_at_k(topk_queries, pos_set, k)
        p_sum += p
        r_sum += r
        f1_sum += f1
        total += 1

    if total == 0:
        return 0.0, 0.0, 0.0

    return p_sum/total, r_sum/total, f1_sum/total


for k in [3, 5, 10, 20]:
    p, r, f1 = eval_model_at_k(k)
    print(f"HybridSPARQLEncoder @ {k:>2} | P: {p:.4f} | R: {r:.4f} | F1: {f1:.4f}")


HybridSPARQLEncoder @  3 | P: 0.1180 | R: 0.0955 | F1: 0.0772
HybridSPARQLEncoder @  5 | P: 0.1323 | R: 0.1392 | F1: 0.0980
HybridSPARQLEncoder @ 10 | P: 0.1289 | R: 0.2086 | F1: 0.1207
HybridSPARQLEncoder @ 20 | P: 0.1155 | R: 0.2871 | F1: 0.1358
